<a href="https://colab.research.google.com/github/dmsejn4/TSD-sequential-adaptation/blob/seed0/scenario2_fr17_seed0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 환경 세팅 및 원본 데이터 준비

In [ ]:
# [1-1] 환경 세팅 및 YOLO-TS 클론
!nvidia-smi
!git clone https://github.com/Heqiang-Huang/YOLO-TS.git
!pip install -q ultralytics
!pip install -r /content/YOLO-TS/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
from pathlib import Path

# [1-2] 원본 45개 클래스 데이터셋 압축 해제 (경로는 실제 파일명에 맞게 수정해주세요)
# 주의: 예전 3class mapped zip 파일이 아닌, GenTT100K 원본 파일이어야 합니다.
zip_path = '/content/drive/MyDrive/GenTT100K_45cls_split(20%).zip'
extract_path = '/content/GenTT100K-yolo'

print("\n🚀 원본 45-class 데이터셋 압축 해제 시작...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print(f"✅ 압축 해제 완료: {extract_path}")

Tue Aug  4 13:04:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# train val 분리

In [ ]:
import os
import random
import shutil
from pathlib import Path

# 📌 압축 푼 최상위 폴더 경로 (train, test 폴더가 있는 곳)
dataset_path = Path('/content/GenTT100K-yolo')

# 경로 설정 (이미지와 라벨이 같은 폴더에 있음)
dir_train = dataset_path / 'train'
dir_val = dataset_path / 'val'

# val 폴더 만들기 (없으면 생성)
dir_val.mkdir(parents=True, exist_ok=True)

# 시드 고정
random.seed(0)

# train 폴더 안에서 이미지 파일(.jpg, .png 등)만 싹 긁어오기
image_files = [f for f in dir_train.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
total_train_before = len(image_files)

# 정확히 20% 수량 계산
val_count = int(total_train_before * 0.2)

# 랜덤하게 20% 이미지 파일 경로 뽑기
val_images = random.sample(image_files, val_count)

print(f"🚀 총 {total_train_before}장 중 20%인 {val_count}장을 VAL 폴더로 쪼개는 중...")

# 선택된 파일들을 val 폴더로 '이동 (Move)'
moved_labels = 0
for img_path in val_images:
    # 1. 이미지 파일 이동
    shutil.move(str(img_path), str(dir_val / img_path.name))

    # 2. 이름이 똑같은 라벨 파일(.txt) 찾아서 이동
    label_path = dir_train / f"{img_path.stem}.txt"
    if label_path.exists():
        shutil.move(str(label_path), str(dir_val / label_path.name))
        moved_labels += 1

print("-" * 45)
print(f"✅ 깔끔하게 쪼개기 완료!")
print(f"  - 이동된 이미지 : {val_count}장")
print(f"  - 이동된 라벨   : {moved_labels}개")
print("-" * 45)
print(f"📁 남은 TRAIN 장수 : {total_train_before - val_count}장 (약 90%)")
print(f"📁 새로 생긴 VAL 장수 : {val_count}장 (약 10%)")

In [ ]:
import shutil
import os

# 📌 1. 압축할 대상 폴더 (아까 train/val/test가 모두 들어있는 최상위 폴더 경로)
# 예: dataset_path = '/content/GenTT100K-yolo'
dataset_path = '/content/GenTT100K-yolo'

# 📌 2. 내 구글 드라이브에 저장될 압축 파일의 이름과 경로
# 끝에 .zip은 안 적어도 돼! 알아서 붙음.
drive_save_path = '/content/drive/MyDrive/GenTT100K_45cls_split(20%)'

print("📦 데이터셋 압축을 시작합니다. (파일이 많아서 1~2분 정도 걸릴 수 있어!)...")

# 3. 폴더 전체를 zip으로 압축해서 드라이브로 바로 저장
shutil.make_archive(
    base_name=drive_save_path, # 저장될 파일 경로 (확장자 제외)
    format='zip',              # 압축 형식
    root_dir=dataset_path      # 압축할 원본 폴더
)

print("-" * 45)
print(f"✅ 드라이브 영구 저장 완료!")
print(f"📂 저장된 경로: {drive_save_path}.zip")
print("-" * 45)
print("💡 꿀팁: 다음부터는 코랩 켤 때마다 이 파일 하나만 압축 풀면 바로 학습 준비 끝이야!")

In [ ]:
import zipfile
from collections import Counter

# 📌 구글 드라이브에 저장한 압축 파일 경로 (확장자 .zip 포함)
zip_path = '/content/drive/MyDrive/GenTT100K_45cls_split(20%).zip'

splits = ['train', 'val', 'test']
# split별로 클래스 ID가 등장한 이미지 장수를 기록할 딕셔너리
class_counts = {split: Counter() for split in splits}

print(f"🔍 [{zip_path}] 압축 파일 스캔 중... 클래스별 이미지 장수를 집계합니다.")
print("💡 압축을 풀지 않고 읽기 때문에 파일이 많아도 빠르게 처리됩니다.\n")

try:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()

        # 1. 각 split별로 존재하는 .txt 라벨 파일 목록 분류
        split_txt_files = {split: [] for split in splits}
        for file_name in file_list:
            if file_name.lower().endswith('.txt'):
                for split in splits:
                    # 폴더 경로에 split 이름이 포함되어 있는지 확인
                    if f"/{split}/" in file_name or file_name.startswith(f"{split}/"):
                        split_txt_files[split].append(file_name)
                        break

        # 2. 각 라벨 파일 내부를 파싱하여 클래스 집계
        for split in splits:
            txt_files = split_txt_files[split]
            print(f"📦 [{split.upper()}] 라벨 분석 중 ({len(txt_files):,}개)...")

            for txt_file in txt_files:
                try:
                    # 한 이미지에 같은 클래스가 여러 개 있을 수 있으므로,
                    # '중복 없이 어떤 클래스들이 포함되어 있는지'만 set으로 추출 (이미지 장수 기준)
                    classes_in_image = set()

                    with zip_ref.open(txt_file) as f:
                        for line in f:
                            line_str = line.decode('utf-8').strip()
                            if line_str:
                                parts = line_str.split()
                                if parts:
                                    # YOLO 포맷의 첫 번째 값은 클래스 ID (정수형 변환)
                                    class_id = int(parts[0])
                                    classes_in_image.add(class_id)

                    # 이 이미지에 등장한 클래스들의 카운트를 1씩 증가
                    for class_id in classes_in_image:
                        class_counts[split][class_id] += 1

                except Exception:
                    # 빈 라벨 파일이거나 인코딩 오류가 발생하는 파일은 안전하게 패스
                    pass

    # 3. 최종 결과 깔끔하게 출력
    print("\n" + "=" * 50)
    print("📊 [각 세트별 클래스별 이미지 분포 결과]")
    print("=" * 50)

    for split in splits:
        print(f"\n📂 [{split.upper()}] 세트 클래스 분포")
        print(f"{'클래스 ID':<10} | {'포함된 이미지 장수':<15}")
        print("-" * 35)

        sorted_classes = sorted(class_counts[split].keys())
        if not sorted_classes:
            print("  (감지된 라벨 데이터가 없습니다)")
        else:
            for cls in sorted_classes:
                count = class_counts[split][cls]
                print(f"Class {cls:<5} | {count:>12,} 장")
        print("-" * 35)

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다! 경로가 맞는지, 드라이브 마운트는 되어 있는지 확인해 주세요.")

# 3. 데이터 YAML 파일 생성

In [ ]:
import yaml

# 📌 1. 네가 깃허브에서 찾아낸 진짜 45개 클래스 이름
real_class_names = [
    "i2", "i4", "i5", "il100", "il60", "il80", "io", "ip", "p10", "p11",
    "p12", "p19", "p23", "p26", "p27", "p3", "p5", "p6", "pg", "ph4",
    "ph4.5", "ph5", "pl100", "pl120", "pl20", "pl30", "pl40", "pl5", "pl50",
    "pl60", "pl70", "pl80", "pm20", "pm30", "pm55", "pn", "pne", "po", "pr40",
    "w13", "w32", "w55", "w57", "w59", "wo"
]

# 📌 2. 우리가 새로 만들 YAML 데이터 구성
yaml_data = {
    'path': '/content/GenTT100K-yolo',
    'train': 'train',
    'val': 'val',
    'test': 'test',
    'nc': 45,
    # 리스트를 YOLO가 좋아하는 {0: 'i2', 1: 'i4', ...} 형태의 딕셔너리로 자동 변환해서 꽂아줌
    'names': {i: name for i, name in enumerate(real_class_names)}
}

# 📌 3. 파일로 저장
yaml_path = '/content/YOLO-TS/GenTT100K_45class.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print(f"✅ 실제 라벨명 완벽 적용! 데이터 YAML 생성 완료: {yaml_path}")

✅ 실제 라벨명 완벽 적용! 데이터 YAML 생성 완료: /content/YOLO-TS/GenTT100K_45class.yaml


# 4. 전이학습 실행

In [ ]:
import gc
import torch
import os
import sys
import re

# 🚨 PyTorch 2.6 보안 에러 우회 (가장 깔끔한 원본 버전)
original_load = torch.load
def safe_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = safe_load

# 메모리 정리 및 환경 설정
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# 로컬 YOLO-TS 폴더 연결
sys.path.insert(0, '/content/YOLO-TS')
from ultralytics import YOLO

# 1. 모델 설정 파일(Config) 변경: CCTSDB2021 -> TT100K
src_yaml = "/content/YOLO-TS/YOLO-TS_TT100K.yaml"
new_yaml = "/content/YOLO-TS/YOLO-TS_45cls.yaml"

with open(src_yaml, "r") as f:
    text = f.read()

# TT100K 원본 YAML의 클래스 수를 무조건 45로 덮어쓰기
text = re.sub(r'nc:\s*\d+', 'nc: 45', text)

with open(new_yaml, "w") as f:
    f.write(text)

# 2. 사전학습 가중치 파일 경로 및 데이터 YAML 경로
best_path = "/content/drive/MyDrive/TT100K_best.pt"
data_path = "/content/YOLO-TS/GenTT100K_45class.yaml"

# 3. 45개의 클래스를 가진 모델 구조 초기화
model = YOLO(new_yaml)

# 4. 기존 학습된 가중치 불러오기
model.load(best_path)

print("🚀 TT100K 기반 45-class 전이학습 시작 (특징 추출기 Freezing 적용)")

# 5. 전이학습 실행
results = model.train(
        data=data_path,
        imgsz=640,              # 비교 위해 고정
        epochs=150,
        patience=15,
        batch=16,
        nbs=64,

        # --- optimizer ---
        optimizer="AdamW",
        lr0=0.0005,
        lrf=0.01,
        weight_decay=0.01,
        cos_lr=True,
        warmup_epochs=5.0,

        # --- augmentation (명시적 지정) ---
        mosaic=1.0,
        close_mosaic=20,
        mixup=0.15,
        copy_paste=0.1,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.0,             # 표지판 방향성 보존
        flipud=0.0,

        # --- misc ---
        seed=0,
        deterministic=True,
        freeze=17,
        save_period=10,
        project='/content/drive/MyDrive/runs',
        name='scenario3_fr17',
        amp=True,
        workers=2,
        cache=False,  # 🔥 형태 보존을 위해 off
)

print("\n✅ TT100K 사전학습 모델 기반 전이학습 완료!")
from google.colab import runtime

runtime.unassign()

WARNING ⚠️ no model scale passed. Assuming scale='l'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  3     70272  ultralytics.nn.modules.block.C2f             [64, 64, 3, True]             
  2                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  3                  -1  1    115456  ultralytics.nn.modules.block.C2f             [128, 128, 1, True]           
  4                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  5                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  6                  -1  1   1180672  ultralytics.nn.modules.conv.Conv             [256, 512, 3, 2]              
  7                  -1  1   1838

🚀 TT100K 기반 45-class 전이학습 시작 (특징 추출기 Freezing 적용)


Ultralytics YOLOv8.0.180 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=/content/YOLO-TS/YOLO-TS_45cls.yaml, data=/content/YOLO-TS/GenTT100K_45class.yaml, epochs=150, patience=15, batch=16, imgsz=640, save=True, save_period=10, cache=False, device=None, workers=2, project=/content/drive/MyDrive/runs, name=scenario3_fr17, exist_ok=False, pretrained=False, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=20, resume=False, amp=True, fraction=1.0, profile=False, freeze=17, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, show=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, vid_stride=1, stream_buffer=False, line_width=None, visualize=False, augment=False, agnostic_nms=False,


✅ TT100K 사전학습 모델 기반 전이학습 완료!


In [ ]:
# 1. 방금 학습이 완료된 최고 성능 모델 불러오기
saved_model_path = "/content/drive/MyDrive/runs/GenTT100K_transfer_150epoch_final/weights/best.pt"
val_model = YOLO(saved_model_path)

print("🚀 rect=False 모드로 검증(Validation) 재시작")

# 2. 직사각형 검증(rect=True) 비활성화 후 평가 진행
metrics = val_model.val(
    data=data_path,
    imgsz=640,
    batch=16,
    rect=False,    # 💡 핵심: 1픽셀 어긋남 방지를 위해 정사각형 고정
    workers=2
)

Ultralytics YOLOv8.0.180 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO-TS_45cls summary (fused): 127 layers, 11022397 parameters, 0 gradients, 98.4 GFLOPs


🚀 rect=False 모드로 검증(Validation) 재시작


val: Scanning /content/GenTT100K-yolo/val.cache... 1221 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1221/1221 [00:00<?, ?it/s]
val: WARNING ⚠️ /content/GenTT100K-yolo/val/62778.jpg: 2 duplicate labels removed
val: WARNING ⚠️ /content/GenTT100K-yolo/val/90422.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 77/77 [00:10<00:00,  7.35it/s]
                   all       1221       3180      0.855      0.817      0.871      0.674
                    i2       1221         68      0.873      0.779      0.838      0.595
                    i4       1221        102      0.878      0.794      0.867       0.62
                    i5       1221        233      0.918      0.906      0.944      0.642
                 il100       1221         18      0.808          1      0.982      0.819
                  il60       1221         64      0.944      0.984      0.983      0.822
                  il80       1

# 5. 학습 결과 확인 및 평가

In [ ]:
import os

# 가장 최근에 학습된 폴더 찾기
runs_path = '/content/drive/MyDrive/runs'
folders = [f for f in os.listdir(runs_path) if f.startswith('scenario3_fr17')]
folders.sort()

if folders:
    best_weight = f"{runs_path}/{folders[-1]}/weights/best.pt"
    print(f"✅ 평가할 모델: {best_weight}")

    # 모델 로드 후 Test Set 평가
    trained_model = YOLO(best_weight)
    test_metrics = trained_model.val(
        data='/content/YOLO-TS/GenTT100K_45class.yaml',
        split='test',
        imgsz=640,
        batch=16,
        conf=0.001,
        iou=0.6
    )

    print("\n[Test Set 전체 성능]")
    print(f"  mAP50:     {test_metrics.box.map50:.5f}")
    print(f"  mAP50-95:  {test_metrics.box.map:.5f}")
else:
    print("❌ 학습 폴더를 찾을 수 없습니다.")

✅ 평가할 모델: /content/drive/MyDrive/runs/scenario3_fr17/weights/best.pt


Ultralytics YOLOv8.0.180 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO-TS_45cls summary (fused): 127 layers, 11022397 parameters, 0 gradients, 98.4 GFLOPs
100%|██████████| 755k/755k [00:00<00:00, 24.4MB/s]
val: Scanning /content/GenTT100K-yolo/test... 3071 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3071/3071 [00:03<00:00, 889.94it/s] 
val: WARNING ⚠️ /content/GenTT100K-yolo/test/78585.jpg: 1 duplicate labels removed
val: WARNING ⚠️ /content/GenTT100K-yolo/test/79029.jpg: 2 duplicate labels removed
val: New cache created: /content/GenTT100K-yolo/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 192/192 [01:49<00:00,  1.76it/s]
                   all       3071       7706      0.771      0.601      0.692      0.499
                    i2       3071        142      0.797      0.514      0.622      0.413
                    i4       3071        231      0.857      0.624      0.768      0.486



[Test Set 전체 성능]
  mAP50:     0.69198
  mAP50-95:  0.49865


In [ ]:
import sys
import torch

# 1. 🚨 PyTorch 2.6 보안 에러 우회 패치 (런타임 재시작 시 필수!)
original_load = torch.load
def safe_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = safe_load

# 2. 로컬 YOLO-TS 폴더 연결 및 YOLO 임포트
sys.path.insert(0, '/content/YOLO-TS')
from ultralytics import YOLO

# 3. 평가할 전이학습 완료 모델 가중치 경로
best_weight = '/content/drive/MyDrive/runs/scenario3_fr17/weights/best.pt'

# 4. 모델 로드
print(f"🚀 테스트할 모델 로딩 중: {best_weight}")
trained_model = YOLO(best_weight)


🚀 테스트할 모델 로딩 중: /content/drive/MyDrive/runs/scenario3_fr17/weights/best.pt
